# Multimodal Emotion Detection - Fusion of ResNet50 and DistilBERT

## Imports

In [1]:


import os
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image
import pandas as pd
import numpy as np
from transformers import DistilBertTokenizer, DistilBertModel
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter


C:\Users\USER\PycharmProjects\DSGP15_Project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Paths & Device

In [2]:
ROOT_DIR = r"C:\Users\USER\PycharmProjects\DSGP15_Project\ml-models\dataset\Dataset"

train_image_dir = os.path.join(ROOT_DIR, "Images", "Emotion", "train")
test_image_dir = os.path.join(ROOT_DIR, "Images", "Emotion", "test")

bert_model_dir = os.path.join(ROOT_DIR, "DistilBERT", "saved_emotion_bert")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


DistilBertModel(
  (embeddings): Embeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (transformer): Transformer(
    (layer): ModuleList(
      (0-5): 6 x TransformerBlock(
        (attention): DistilBertSdpaAttention(
          (dropout): Dropout(p=0.1, inplace=False)
          (q_lin): Linear(in_features=768, out_features=768, bias=True)
          (k_lin): Linear(in_features=768, out_features=768, bias=True)
          (v_lin): Linear(in_features=768, out_features=768, bias=True)
          (out_lin): Linear(in_features=768, out_features=768, bias=True)
        )
        (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
        (ffn): FFN(
          (dropout): Dropout(p=0.1, inplace=False)
          (lin1): Linear(in_features=768, out_features=3072, bias=True)
          (lin2): L

## Load Dataset CSVs

In [3]:
EMOTION_TRAIN_CSV_PATH = os.path.join(ROOT_DIR, "Texts", "Emotion", "Emotion_Train.csv")
EMOTION_TEST_CSV_PATH = os.path.join(ROOT_DIR, "Texts", "Emotion", "Emotion_Test.csv")

# Load CSVs
emo_train = pd.read_csv(EMOTION_TRAIN_CSV_PATH, header=None)
emo_train.columns = ['ImageID', 'Desc_Turk', 'Desc_Eng', 'Emotion', 'Invalid']
mask = emo_train['Invalid'].notnull()
emo_train.loc[mask, 'Desc_Eng'] = emo_train.loc[mask, 'Invalid']
emo_train = emo_train.drop(columns=['Invalid'])

emo_test = pd.read_csv(EMOTION_TEST_CSV_PATH, header=None)
emo_test.columns = ['ImageID', 'Desc_Turk', 'Desc_Eng', 'Emotion']

# Encode emotion labels
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
emo_train['label'] = le.fit_transform(emo_train['Emotion'])
emo_test['label'] = le.transform(emo_test['Emotion'])

# Split train into train/val
train_df, val_df = train_test_split(emo_train, test_size=0.2, random_state=42)


## Image Transformations

In [4]:


IMG_SIZE = 224
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])


Text feature shape: torch.Size([1, 768])


## Load ResNet50 Feature Extractor

In [ ]:
resnet = models.resnet50(pretrained=True)
resnet.fc = nn.Identity()  # Remove classifier
resnet = resnet.to(device)
resnet.eval()


## Load DistilBERT Feature Extractor

In [ ]:
tokenizer = DistilBertTokenizer.from_pretrained(bert_model_dir)
bert = DistilBertModel.from_pretrained(bert_model_dir)
bert.eval()


## Function to extract text features

In [ ]:
def extract_text_features(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding="max_length", max_length=64)
    with torch.no_grad():
        outputs = bert(**inputs)
    return outputs.last_hidden_state[:, 0, :]  # CLS token


## Test prediction

In [ ]:

test_image = "../Images/Emotion/test/happy/child1.jpg"
test_text = "The child looks very happy and excited"

prediction = predict_emotion_multimodal(test_image, test_text)
print("Predicted emotion:", prediction)
